# Walk-forward folds and market regimes

The frozen v1 core fits nothing across runs, so an expanding
walk-forward reduces to **one** full-history run whose equity curve is
sliced into calendar folds (`amat.validation.walkforward`). Fold
dispersion measures *consistency* — whether the aggregate was earned
across years or granted by one of them. It is **not** out-of-sample
evidence: this configuration was frozen after the history below was
known; untouched data begins only after the declared holdout boundary
(`outputs/holdouts/`).

*Backtest research only. The record-persisting entry point is
`amat walkforward`; this notebook demonstrates the same machinery on a
reduced window.*

In [1]:
# Locate the repository root whether this kernel started in notebooks/ or the root.
import datetime as dt
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "configs").exists() else Path.cwd().parent
assert (ROOT / "configs" / "strategy" / "amat_v1.toml").exists(), ROOT

import matplotlib
matplotlib.use("Agg")  # headless-safe; inline display still works
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from amat.config.loading import load_strategy_config
from amat.data.snapshot import load_corporate_actions, load_panel

SNAPSHOT = ROOT / "data" / "snapshots" / "alpaca-sip-2026-08-11"
resolved = load_strategy_config(ROOT / "configs" / "strategy" / "amat_v1.toml")
strategy = resolved.strategy
symbols = [str(s) for s in strategy.universe.symbols]

# A reduced window keeps every notebook run affordable (minutes, not an hour).
# The CLI commands run the full sample and persist run records; this notebook
# demonstrates the same machinery through the library API.
START = dt.date(2021, 1, 4)
panel, manifest = load_panel(SNAPSHOT, symbols, start=START)
actions = load_corporate_actions(SNAPSHOT, symbols, sessions=panel.sessions)
print(f"snapshot {manifest.snapshot_id}: {len(panel)} sessions from {panel.sessions[0]}")
print(f"config {resolved.content_sha256[:16]}  strategy {strategy.name} v{strategy.version}")


snapshot alpaca-sip-2026-08-11: 1406 sessions from 2021-01-04
config ea8cbd5ad7418a38  strategy amat-etf v1.0.0


In [2]:
from decimal import Decimal
from amat.validation.walkforward import walk_forward

result = walk_forward(
    strategy,
    panel,
    strategy.to_universe(),
    initial_cash=Decimal("100000"),
    actions_by_session=actions,
)
folds = pl.DataFrame([
    {
        "fold": fold.label,
        "sessions": fold.statistics.sessions,
        "total_return": round(fold.statistics.total_return, 4),
        "annualized": round(fold.statistics.annualized_return, 4),
        "volatility": round(fold.statistics.annualized_volatility, 4),
        "sharpe": round(fold.statistics.sharpe, 2),
        "max_drawdown": round(fold.statistics.max_drawdown, 4),
        "trades": fold.statistics.trades,
    }
    for fold in result.folds
])
print(folds)
print()
print(f"positive folds: {result.positive_fold_fraction:.0%}")
print(f"full sample: sharpe {result.full_sample.sharpe:.2f}, "
      f"max drawdown {result.full_sample.max_drawdown:.2%}")
print()
print("!", result.caveat)

shape: (5, 8)
┌───────────┬──────────┬──────────────┬────────────┬────────────┬────────┬──────────────┬────────┐
│ fold      ┆ sessions ┆ total_return ┆ annualized ┆ volatility ┆ sharpe ┆ max_drawdown ┆ trades │
│ ---       ┆ ---      ┆ ---          ┆ ---        ┆ ---        ┆ ---    ┆ ---          ┆ ---    │
│ str       ┆ i64      ┆ f64          ┆ f64        ┆ f64        ┆ f64    ┆ f64          ┆ i64    │
╞═══════════╪══════════╪══════════════╪════════════╪════════════╪════════╪══════════════╪════════╡
│ 2021+2022 ┆ 252      ┆ 0.0173       ┆ 0.0174     ┆ 0.0658     ┆ 0.26   ┆ -0.063       ┆ 223    │
│ 2023      ┆ 251      ┆ 0.0219       ┆ 0.0221     ┆ 0.0556     ┆ 0.4    ┆ -0.0624      ┆ 428    │
│ 2024      ┆ 253      ┆ 0.072        ┆ 0.072      ┆ 0.0748     ┆ 0.96   ┆ -0.0483      ┆ 461    │
│ 2025      ┆ 251      ┆ 0.0998       ┆ 0.1007     ┆ 0.0672     ┆ 1.5    ┆ -0.0526      ┆ 431    │
│ 2026      ┆ 152      ┆ 0.1084       ┆ 0.1874     ┆ 0.0891     ┆ 2.1    ┆ -0.0429      ┆ 264  

In [3]:
fig, ax = plt.subplots(figsize=(8, 3.5))
labels = [fold.label for fold in result.folds]
sharpes = [fold.statistics.sharpe for fold in result.folds]
colors = ["#2f7fd1" if s >= 0 else "#c62828" for s in sharpes]
ax.bar(labels, sharpes, color=colors)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Per-fold Sharpe - consistency, not out-of-sample proof")
ax.grid(axis="y", alpha=0.3)
plt.show()

for gate in result.research_gates():
    status = "PASS" if gate.passed else ("FAIL" if gate.measurable else "NOT MEASURABLE")
    print(f"[{status}] {gate.name}: {gate.measured} (limit {gate.limit})")
print("(necessary, never sufficient - see .claude/references/promotion-gates.md)")

[PASS] aggregate net Sharpe at least 0.50 (in-sample analogue of the out-of-sample gate): 0.99 (limit >= 0.50)
[PASS] at least 60% positive annual folds (in-sample analogue of the out-of-sample gate): 100% (limit >= 60% across >= 4 folds)
[PASS] max drawdown no worse than 20% (in-sample analogue of the out-of-sample gate): -7.86% (limit >= -20%)
(necessary, never sufficient - see .claude/references/promotion-gates.md)


/tmp/claude-1000/ipykernel_119/2820450998.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Performance by market regime

Sessions are labelled causally from SPY's own past — trailing 63-session
return sign (bull/bear) × trailing 21-session realized volatility
against its expanding median (quiet/volatile). The labels condition
nothing; they only describe where the run's session returns landed.
Session `t`'s return is booked under session `t-1`'s label: the label
at `t` contains `t`'s own move, and the contemporaneous convention
manufactures a ~1.27× quiet/volatile separation on pure noise.
Per-regime drawdowns are undefined (the sessions are interleaved) and
deliberately absent.

In [4]:
from amat.validation.regimes import classify_sessions, regime_performance

labels = classify_sessions(panel)
slices = regime_performance(result.equity_curve, result.session_dates, panel, labels)
regimes = pl.DataFrame([
    {
        "regime": entry.label,
        "sessions": entry.sessions,
        "share": round(entry.share_of_sample, 3),
        "annualized": round(entry.annualized_return, 4),
        "volatility": round(entry.annualized_volatility, 4),
        "sharpe": round(entry.sharpe, 2),
        "hit_rate": round(entry.hit_rate, 2),
    }
    for entry in slices
])
print(regimes)

shape: (4, 7)
┌───────────────┬──────────┬───────┬────────────┬────────────┬────────┬──────────┐
│ regime        ┆ sessions ┆ share ┆ annualized ┆ volatility ┆ sharpe ┆ hit_rate │
│ ---           ┆ ---      ┆ ---   ┆ ---        ┆ ---        ┆ ---    ┆ ---      │
│ str           ┆ i64      ┆ f64   ┆ f64        ┆ f64        ┆ f64    ┆ f64      │
╞═══════════════╪══════════╪═══════╪════════════╪════════════╪════════╪══════════╡
│ bull-quiet    ┆ 599      ┆ 0.519 ┆ 0.0732     ┆ 0.0746     ┆ 0.98   ┆ 0.56     │
│ bull-volatile ┆ 215      ┆ 0.186 ┆ 0.1018     ┆ 0.0588     ┆ 1.73   ┆ 0.59     │
│ bear-quiet    ┆ 65       ┆ 0.056 ┆ -0.1044    ┆ 0.0437     ┆ -2.39  ┆ 0.46     │
│ bear-volatile ┆ 275      ┆ 0.238 ┆ 0.0803     ┆ 0.0712     ┆ 1.13   ┆ 0.55     │
└───────────────┴──────────┴───────┴────────────┴────────────┴────────┴──────────┘


## Caveats, restated

- One frozen configuration, one vendor's data, one decade. Fold and
  regime tables describe this sample; they do not forecast.
- The 2021 start date here exists to keep the notebook fast; the
  canonical full-sample numbers live in the persisted run records under
  `outputs/runs/` and in `docs/experiment-registry.md`.
- Regime conditioning as a *strategy input* is a registered experiment
  (`amat regime-experiment`) judged by falsification controls — see the
  registry for verdicts before drawing any conclusion from these
  slices.